In [2]:
pip install open_clip_torch --upgrade

  Obtaining dependency information for open_clip_torch from https://files.pythonhosted.org/packages/32/f9/0458745c1d299411161ee3b6c32228a3de0be1d8497d779fd7f17a8e96aa/open_clip_torch-2.32.0-py3-none-any.whl.metadata
  Obtaining dependency information for timm from https://files.pythonhosted.org/packages/6c/d0/179abca8b984b3deefd996f362b612c39da73b60f685921e6cd58b6125b4/timm-1.0.15-py3-none-any.whl.metadata
     ---------------------------------------- 0.0/52.0 kB ? eta -:--:--
     ---------------------------------------- 52.0/52.0 kB 1.3 MB/s eta 0:00:00
   ---------------------------------------- 0.0/1.5 MB ? eta -:--:--
   ------ --------------------------------- 0.3/1.5 MB 5.4 MB/s eta 0:00:01
   ---------------------------------------  1.5/1.5 MB 16.3 MB/s eta 0:00:01
   ---------------------------------------- 1.5/1.5 MB 13.9 MB/s eta 0:00:00
   ---------------------------------------- 0.0/2.4 MB ? eta -:--:--
   ---------------------------------------  2.4/2.4 MB 73.3 MB/s eta 0

In [3]:
import pandas as pd, torch
from torch.utils.data import Dataset
from torchvision import transforms as T
from PIL import Image
from open_clip import tokenizer as clip_tokenizer

transform = T.Compose([
    T.Resize(224, antialias=True),
    T.CenterCrop(224),
    T.ToTensor(),
    T.Normalize((0.48145466,0.4578275,0.40821073),
                (0.26862954,0.26130258,0.27577711)),
])

C:\Users\steph\anaconda3\Lib\site-packages\transformers\utils\generic.py:260: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  torch.utils._pytree._register_pytree_node(


In [6]:
class RetrievalDataset(Dataset):
    def __init__(self, parquet_path= r"C:\Users\steph\OneDrive\Desktop\data\metadata.parquet",
                 split="train", transform=transform):
        self.df = pd.read_parquet(parquet_path)
        self.df = self.df[self.df.split == split].reset_index(drop=True)
        self.tok = clip_tokenizer.SimpleTokenizer()
        self.transform = transform
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        r = self.df.iloc[idx]
        img = self.transform(Image.open(r.image_path).convert("RGB"))
        txt = self.tok(r.caption)
        return {"image": img, "text": txt, "domain": r.domain, "id": r.id}

In [7]:
ds = RetrievalDataset()
print("Dataset size:", len(ds))
ex = ds[0]
ex["image"].shape, ex["text"][:10], ex["domain"]

Dataset size: 850668


(torch.Size([3, 224, 224]),
 tensor([[49406,  3086,   539,   674,  5508, 20414,  9920,   550, 17143, 49407,
              0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
              0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
              0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
              0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
              0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
              0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
              0,     0,     0,     0,     0,     0,     0]]),
 'sd')